# Data Integration Phase
Enriches data from silver lake and writes aggregated and enriched taxi_trips data as gold Delta table `integrated_taxi_trips`.


## 1. Configure Spark


In [1]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.spark import create_spark, project_root

spark = create_spark("integration-gold")
ROOT = project_root()

from src.lake import GOLD, SILVER, read_delta, show_delta, write_gold

print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("PySpark:", spark.version)
print("ROOT:", ROOT)


:: loading settings :: url = jar:file:/Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/samuelflodin/.ivy2/cache
The jars for the packages stored in: /Users/samuelflodin/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-aa991233-1c69-4d86-9f69-07f5561425e5;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 130ms :: artifacts dl 4ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     

JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home
PySpark: 3.5.4
ROOT: /Users/samuelflodin/programmering/uppgifter/5an/bigdata/id2221-labs


## 2. Load silver tables

Read silver delta tables.

In [2]:
from pyspark.sql import functions as F

trips = read_delta(spark, SILVER / "taxi_trips")
weather = read_delta(spark, SILVER / "weather")
air_quality = read_delta(spark, SILVER / "air_quality")
zones = read_delta(spark, SILVER / "taxi_zones")

## 3. Hourly weather (NYC local)



In [4]:
hourly_weather = (
    weather
    .groupBy(
        F.col("observation_date").alias("pickup_date"),
        F.col("observation_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("temperature_c").alias("temperature_c"),
        F.avg("wind_speed_ms").alias("wind_speed_ms"),
    )
)

print(f"hourly_weather: {hourly_weather.count():,} hours")
hourly_weather.show(3, truncate=False)


26/09/18 18:32:04 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


hourly_weather: 5,681 hours


+-----------+-----------+-------------+-------------+
|pickup_date|pickup_hour|temperature_c|wind_speed_ms|
+-----------+-----------+-------------+-------------+
|2025-02-16 |5          |1.7          |4.1          |
|2025-02-16 |14         |3.3          |0.75         |
|2025-02-16 |23         |2.8          |6.7          |
+-----------+-----------+-------------+-------------+
only showing top 3 rows



## 4. Hourly air quality (NYC local)



In [5]:
NYC_COUNTIES = [5, 47, 61, 81, 85]

hourly_aq = (
    air_quality
    .filter(F.col("county_code").isin(NYC_COUNTIES))
    .groupBy(
        F.col("measurement_date").alias("pickup_date"),
        F.col("measurement_hour").alias("pickup_hour"),
    )
    .agg(
        F.avg("value").alias("pm25"),
        F.first("unit").alias("pm25_unit"),
    )
)

print(f"hourly_aq: {hourly_aq.count():,} hours")
hourly_aq.show(3, truncate=False)


hourly_aq: 8,759 hours


+-----------+-----------+------------------+---------------------------+
|pickup_date|pickup_hour|pm25              |pm25_unit                  |
+-----------+-----------+------------------+---------------------------+
|2025-01-01 |0          |13.55             |Micrograms/cubic meter (LC)|
|2025-01-01 |1          |10.7625           |Micrograms/cubic meter (LC)|
|2025-01-01 |2          |11.912500000000001|Micrograms/cubic meter (LC)|
+-----------+-----------+------------------+---------------------------+
only showing top 3 rows



## 5. Pickup / dropoff zone lookups

`taxi_zones` is a small table ==> we can easily and with minimal overhead split them into `pickup_zones` and `dropoff_zones`


In [6]:
pickup_zones = zones.select(
    F.col("location_id").alias("pickup_location_id"),
    F.col("zone").alias("pickup_zone"),
    F.col("borough").alias("pickup_borough"),
)

dropoff_zones = zones.select(
    F.col("location_id").alias("dropoff_location_id"),
    F.col("zone").alias("dropoff_zone"),
    F.col("borough").alias("dropoff_borough"),
)


## 6. Enrich trips and write `integrated_taxi_trips`

Left-join weather and air_quality data onto taxi_trip data.

In [7]:
integrated = (
    trips
    .join(F.broadcast(pickup_zones), "pickup_location_id", "left")
    .join(F.broadcast(dropoff_zones), "dropoff_location_id", "left")
    .join(F.broadcast(hourly_weather), ["pickup_date", "pickup_hour"], "left")
    .join(F.broadcast(hourly_aq), ["pickup_date", "pickup_hour"], "left")
    .fillna(
        {
            "pickup_zone": "UNKNOWN",
            "pickup_borough": "UNKNOWN",
            "dropoff_zone": "UNKNOWN",
            "dropoff_borough": "UNKNOWN",
        }
    )
    .select(
        "taxi_type",
        "vendor_id",
        "pickup_datetime",
        "dropoff_datetime",
        "passenger_count",
        "trip_distance",
        "pickup_location_id",
        "pickup_zone",
        "pickup_borough",
        "dropoff_location_id",
        "dropoff_zone",
        "dropoff_borough",
        "fare_amount",
        "tip_amount",
        "tolls_amount",
        "total_amount",
        "temperature_c",
        "wind_speed_ms",
        "pm25",
        "pm25_unit",
        "pickup_date",
        "pickup_hour",
    )
    .cache()
)

# Materialize the shared input once so both writes measure partitioning and I/O.
integrated.count()

import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips", partition_by=["pickup_date"])
print(f"write by_date: {time.perf_counter() - t0:.1f}s")
show_delta(spark, GOLD / "integrated_taxi_trips")

write by_date: 140.3s


integrated_taxi_trips: 18961395 rows @ data/lake/gold/integrated_taxi_trips


+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-------------------+--------------+-------------------+-----------------------+---------------+-----------+----------+------------+------------+------------------+------------------+-----+---------------------------+-----------+-----------+
|taxi_type|vendor_id|pickup_datetime    |dropoff_datetime   |passenger_count|trip_distance|pickup_location_id|pickup_zone        |pickup_borough|dropoff_location_id|dropoff_zone           |dropoff_borough|fare_amount|tip_amount|tolls_amount|total_amount|temperature_c     |wind_speed_ms     |pm25 |pm25_unit                  |pickup_date|pickup_hour|
+---------+---------+-------------------+-------------------+---------------+-------------+------------------+-------------------+--------------+-------------------+-----------------------+---------------+-----------+----------+------------+------------+------------------+------------------+-----+

## 7. Two storage designs

To compare two storage designs, we can partition the same rows aggregated from silver to gold layer in two different ways:

| Partitioning Logic | Table | Partition | Suited for |
| --- | --- | --- | --- |
| By timestamp | `integrated_taxi_trips` | `pickup_date` | average duration per day |
| By location | `integrated_taxi_trips_by_borough` | `pickup_borough` | trip data per borough location |


In [8]:
import time

t0 = time.perf_counter()
write_gold(integrated, "integrated_taxi_trips_by_borough", partition_by=["pickup_borough"])
print(f"write by_borough: {time.perf_counter() - t0:.1f}s")


def storage_report(table_name: str) -> None:
    path = GOLD / table_name
    files = [f for f in path.rglob("*.parquet") if f.is_file()]
    size_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
    n_parts = len({f.parent for f in files})
    print(
        f"{table_name:40} files={len(files):>5}  partitions={n_parts:>4}  size={size_mb:>8.1f} MB"
    )


print()
print("Storage")
storage_report("integrated_taxi_trips")
storage_report("integrated_taxi_trips_by_borough")

write by_borough: 81.2s

Storage
integrated_taxi_trips                    files=  872  partitions= 151  size=   930.3 MB
integrated_taxi_trips_by_borough         files=  464  partitions=   8  size=   912.5 MB


### Queries on both designs


In [9]:
import time


def queries(df):
    duration_min = (
        F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")
    ) / 60.0
    return {
        "trips per borough": (
            df.groupBy("pickup_borough")
            .agg(F.count(F.lit(1)).alias("trips"))
            .orderBy(F.desc("trips"))
        ),
        "avg duration per day": (
            df.withColumn("duration_min", duration_min)
            .groupBy("pickup_date")
            .agg(F.avg("duration_min").alias("avg_duration_min"))
            .orderBy("pickup_date")
        ),
        "avg fare per borough": (
            df.groupBy("pickup_borough")
            .agg(F.avg("fare_amount").alias("avg_fare"))
            .orderBy("pickup_borough")
        ),
    }


def run_queries(table_name: str) -> None:
    spark.catalog.clearCache()
    df = read_delta(spark, GOLD / table_name)
    print(f"\n{table_name}")
    print("-" * 40)
    for name, q in queries(df).items():
        t0 = time.perf_counter()
        rows = q.collect()
        elapsed = time.perf_counter() - t0
        print(f"\n{name}  ({elapsed:.2f}s, {len(rows):,} rows)")
        spark.createDataFrame(rows).show(20, truncate=False)


run_queries("integrated_taxi_trips")
run_queries("integrated_taxi_trips_by_borough")



integrated_taxi_trips
----------------------------------------



trips per borough  (2.26s, 8 rows)


+--------------+--------+
|pickup_borough|trips   |
+--------------+--------+
|Manhattan     |16486585|
|Queens        |1721335 |
|Brooklyn      |570889  |
|Bronx         |133362  |
|Unknown       |38192   |
|N/A           |7454    |
|EWR           |2016    |
|Staten Island |1562    |
+--------------+--------+




avg duration per day  (4.23s, 151 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2025-01-01 |15.609852961742053|
|2025-01-02 |16.915689523832487|
|2025-01-03 |16.065138736299335|
|2025-01-04 |15.06453791156963 |
|2025-01-05 |14.827447703371156|
|2025-01-06 |15.308618129029368|
|2025-01-07 |14.913071977708395|
|2025-01-08 |14.797202941625418|
|2025-01-09 |15.496823451085946|
|2025-01-10 |15.209765819403058|
|2025-01-11 |13.785585752974244|
|2025-01-12 |13.926031191008917|
|2025-01-13 |14.925257468273719|
|2025-01-14 |15.036757210857543|
|2025-01-15 |15.370316278918276|
|2025-01-16 |15.688982134980185|
|2025-01-17 |16.11724896722972 |
|2025-01-18 |14.312338544421385|
|2025-01-19 |12.98765114039252 |
|2025-01-20 |14.207783542364009|
+-----------+------------------+
only showing top 20 rows




avg fare per borough  (2.33s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+------------------+
|Bronx         |27.761668466279758|
|Brooklyn      |26.234144851275925|
|EWR           |86.18024305555555 |
|Manhattan     |15.723564235408409|
|N/A           |89.83212503353904 |
|Queens        |48.197809020324094|
|Staten Island |35.1838796414853  |
|Unknown       |21.12445067029745 |
+--------------+------------------+


integrated_taxi_trips_by_borough
----------------------------------------



trips per borough  (8.14s, 8 rows)
+--------------+--------+
|pickup_borough|trips   |
+--------------+--------+
|Manhattan     |16486585|
|Queens        |1721335 |
|Brooklyn      |570889  |
|Bronx         |133362  |
|Unknown       |38192   |
|N/A           |7454    |
|EWR           |2016    |
|Staten Island |1562    |
+--------------+--------+




avg duration per day  (6.20s, 151 rows)
+-----------+------------------+
|pickup_date|avg_duration_min  |
+-----------+------------------+
|2025-01-01 |15.60985296174202 |
|2025-01-02 |16.91568952383238 |
|2025-01-03 |16.06513873629931 |
|2025-01-04 |15.064537911569811|
|2025-01-05 |14.827447703371112|
|2025-01-06 |15.308618129029233|
|2025-01-07 |14.913071977708357|
|2025-01-08 |14.797202941625548|
|2025-01-09 |15.4968234510859  |
|2025-01-10 |15.209765819403   |
|2025-01-11 |13.78558575297413 |
|2025-01-12 |13.926031191008896|
|2025-01-13 |14.925257468273726|
|2025-01-14 |15.036757210857575|
|2025-01-15 |15.370316278918331|
|2025-01-16 |15.688982134980206|
|2025-01-17 |16.117248967230005|
|2025-01-18 |14.31233854442128 |
|2025-01-19 |12.987651140392456|
|2025-01-20 |14.207783542363842|
+-----------+------------------+
only showing top 20 rows




avg fare per borough  (1.41s, 8 rows)
+--------------+------------------+
|pickup_borough|avg_fare          |
+--------------+------------------+
|Bronx         |27.761668466279733|
|Brooklyn      |26.23414485127596 |
|EWR           |86.18024305555555 |
|Manhattan     |15.723564235404249|
|N/A           |89.83212503353919 |
|Queens        |48.19780902032546 |
|Staten Island |35.18387964148521 |
|Unknown       |21.124450670297374|
+--------------+------------------+



### Query 1

In [10]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_1 = """
WITH monthly_zone_trips AS (
    SELECT
        TRUNC(pickup_date, 'MM') AS trip_month,
        pickup_location_id,
        pickup_borough,
        pickup_zone,
        pickup_date
    FROM integrated_taxi_trips
    WHERE pickup_zone != 'UNKNOWN'
      AND pickup_date IS NOT NULL
)
SELECT
    trip_month,
    pickup_location_id,
    pickup_borough,
    pickup_zone,
    COUNT(*) AS total_trips,
    COUNT(DISTINCT pickup_date) AS active_days,
    ROUND(
        CAST(COUNT(*) AS DOUBLE) / COUNT(DISTINCT pickup_date),
        2
    ) AS avg_daily_trips
FROM monthly_zone_trips
GROUP BY trip_month, pickup_location_id, pickup_borough, pickup_zone
ORDER BY trip_month ASC, total_trips DESC
"""

monthly_demand = spark.sql(query_1)
monthly_demand.show(20, truncate=False)

+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|trip_month|pickup_location_id|pickup_borough|pickup_zone                 |total_trips|active_days|avg_daily_trips|
+----------+------------------+--------------+----------------------------+-----------+-----------+---------------+
|2025-01-01|161               |Manhattan     |Midtown Center              |164179     |31         |5296.1         |
|2025-01-01|237               |Manhattan     |Upper East Side South       |159921     |31         |5158.74        |
|2025-01-01|236               |Manhattan     |Upper East Side North       |152107     |31         |4906.68        |
|2025-01-01|132               |Queens        |JFK Airport                 |137501     |31         |4435.52        |
|2025-01-01|230               |Manhattan     |Times Sq/Theatre District   |120108     |31         |3874.45        |
|2025-01-01|186               |Manhattan     |Penn Station/Madison Sq We

### Query 2

In [13]:
query_2 = """
WITH binned_weather AS (
    SELECT
        trip_distance,
        CASE
            WHEN temperature_c IS NULL THEN 'Unknown'
            WHEN temperature_c < 0 THEN 'Freezing (<0°C)'
            WHEN temperature_c BETWEEN 0 AND 10 THEN 'Cold (0°C to 10°C)'
            WHEN temperature_c BETWEEN 10.01 AND 20 THEN 'Moderate (10°C to 20°C)'
            ELSE 'Warm (>20°C)'
        END AS temp_category,
        CASE
            WHEN wind_speed_ms IS NULL THEN 'Unknown'
            WHEN wind_speed_ms < 2 THEN 'Calm (<2 m/s)'
            WHEN wind_speed_ms BETWEEN 2 AND 6 THEN 'Moderate Wind (2-6 m/s)'
            ELSE 'High Wind (>6 m/s)'
        END AS wind_category
    FROM integrated_taxi_trips
    WHERE trip_distance > 0 AND trip_distance < 100
)
SELECT
    temp_category,
    wind_category,
    COUNT(*) AS trip_count,
    ROUND(AVG(trip_distance), 2) AS avg_distance_miles
FROM binned_weather
GROUP BY temp_category, wind_category
ORDER BY temp_category, wind_category
"""

avg_distance_weather = spark.sql(query_2)
avg_distance_weather.show(20, truncate=False)

+-----------------------+-----------------------+----------+------------------+
|temp_category          |wind_category          |trip_count|avg_distance_miles|
+-----------------------+-----------------------+----------+------------------+
|Cold (0°C to 10°C)     |Calm (<2 m/s)          |1354374   |3.29              |
|Cold (0°C to 10°C)     |High Wind (>6 m/s)     |481882    |3.19              |
|Cold (0°C to 10°C)     |Moderate Wind (2-6 m/s)|5131395   |3.22              |
|Cold (0°C to 10°C)     |Unknown                |673443    |3.19              |
|Freezing (<0°C)        |Calm (<2 m/s)          |408557    |3.21              |
|Freezing (<0°C)        |High Wind (>6 m/s)     |251475    |3.12              |
|Freezing (<0°C)        |Moderate Wind (2-6 m/s)|1854667   |3.12              |
|Freezing (<0°C)        |Unknown                |318667    |3.16              |
|Moderate (10°C to 20°C)|Calm (<2 m/s)          |2038213   |3.58              |
|Moderate (10°C to 20°C)|High Wind (>6 m

### Query 3

In [3]:
read_delta(spark, GOLD / "integrated_taxi_trips").createOrReplaceTempView("integrated_taxi_trips")

query_3 = """
WITH rounded_pm25 AS (
    SELECT
        ROUND(pm25, 0) AS pm25_level,
        pickup_date,
        pickup_hour
    FROM integrated_taxi_trips
    WHERE pm25 IS NOT NULL
      AND pickup_date IS NOT NULL
      AND pickup_hour IS NOT NULL
)
SELECT
    pm25_level,
    COUNT(*) AS trips,
    COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS observed_hours,
    ROUND(
        CAST(COUNT(*) AS DOUBLE)
        / COUNT(DISTINCT struct(pickup_date, pickup_hour)),
        2
    ) AS trips_per_hour
FROM rounded_pm25
GROUP BY pm25_level
ORDER BY pm25_level DESC
"""

pm25_demand = spark.sql(query_3)
pm25_demand.show(34, truncate=False)

26/09/19 12:43:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----------+-------+--------------+--------------+
|pm25_level|trips  |observed_hours|trips_per_hour|
+----------+-------+--------------+--------------+
|34.0      |4188   |1             |4188.0        |
|33.0      |9071   |2             |4535.5        |
|31.0      |5019   |1             |5019.0        |
|30.0      |18438  |3             |6146.0        |
|29.0      |18240  |2             |9120.0        |
|28.0      |17702  |3             |5900.67       |
|27.0      |14835  |2             |7417.5        |
|26.0      |66047  |10            |6604.7        |
|25.0      |18435  |3             |6145.0        |
|24.0      |54765  |10            |5476.5        |
|23.0      |20069  |5             |4013.8        |
|22.0      |54775  |12            |4564.58       |
|21.0      |80355  |17            |4726.76       |
|20.0      |83362  |16            |5210.13       |
|19.0      |79234  |17            |4660.82       |
|18.0      |132230 |31            |4265.48       |
|17.0      |105092 |29         

### Query 4

In [5]:
query = """
WITH trips_with_weather AS (
    SELECT 
        pickup_zone,
        pickup_date,
        pickup_hour,
        NTILE(4) OVER (
            PARTITION BY pickup_zone
            ORDER BY (10 * sqrt(wind_speed_ms) - wind_speed_ms + 10.5) * (33 - temperature_c)
        ) AS weather_condition
    FROM integrated_taxi_trips
    WHERE pickup_zone IS NOT NULL AND pickup_date IS NOT NULL AND pickup_hour IS NOT NULL
),
hourly_demand AS (
    SELECT 
        pickup_zone,
        weather_condition,
        COUNT(1) / COUNT(DISTINCT struct(pickup_date, pickup_hour)) AS trips_per_hour
    FROM trips_with_weather
    GROUP BY pickup_zone, weather_condition
),
pivoted AS (
    SELECT * FROM hourly_demand
    PIVOT (
        ROUND(AVG(trips_per_hour), 2)
        FOR weather_condition IN (1 AS coldest, 2 AS cool, 3 AS warm, 4 AS warmest)
    )
)
SELECT 
    pickup_zone,
    coldest, cool, warm, warmest,
    ROUND(((GREATEST(coldest, cool, warm, warmest) - LEAST(coldest, cool, warm, warmest)) / ((coldest + cool + warm + warmest) / 4.0)) * 100, 2) AS pct_variation
FROM pivoted
WHERE (coldest + cool + warm + warmest) / 4.0 >= 10
ORDER BY pct_variation DESC
"""

spark.sql(query).show(truncate=False)

+-----------------------------+-------+------+------+-------+-------------+
|pickup_zone                  |coldest|cool  |warm  |warmest|pct_variation|
+-----------------------------+-------+------+------+-------+-------------+
|Financial District South     |16.25  |13.0  |12.66 |10.69  |42.28        |
|Financial District North     |26.62  |21.65 |21.07 |18.24  |38.27        |
|Meatpacking/West Village West|49.81  |38.38 |40.34 |34.71  |37.0         |
|Lower East Side              |58.06  |45.25 |48.38 |41.47  |34.35        |
|Little Italy/NoLiTa          |52.69  |41.92 |43.93 |38.29  |32.57        |
|Greenwich Village South      |75.86  |62.62 |63.19 |57.27  |28.72        |
|West Village                 |121.51 |101.08|101.63|92.03  |28.33        |
|Battery Park City            |31.51  |28.26 |26.96 |23.97  |27.24        |
|TriBeCa/Civic Center         |66.51  |58.87 |57.36 |50.75  |27.0         |
|World Trade Center           |25.62  |21.19 |21.69 |19.84  |26.17        |
|East Villag